In [ ]:
# pipeline_together.py

import pandas as pd
import math
from together import Together
import gc



TOGETHER_MODEL =[ "meta-llama/Llama-3.3-70B-Instruct-Turbo"]


import math

def get_logprobs(sentence, client, model):
    """
    Returns logprobs for a single sentence (Together API),
    with robust token-logprob alignment.
    """

    response = client.completions.create(
        model=model,
        prompt=sentence,
        max_tokens=1,
        temperature=0,
        logprobs=1,
        echo=True
    )
    token_logprobs = []
   
    
    
    logprob_data = response.prompt[0].logprobs
    
    tokens = logprob_data.tokens
    logprobs_raw = logprob_data.token_logprobs

    #clean + align tokens safely
    
    for tok, lp in zip(tokens, logprobs_raw):
        if lp is None:
            continue  # skip first token (no context)

        token_logprobs.append((tok, float(lp)))

    
    

    # no valid tokens
    if len(token_logprobs) == 0:
        return {
            "surprisal": None,
            "avg_log_prob": None,
            "acceptability_prob": None,
            "log_probs_per_token": None,
        }
    token_logprobs = [
        (tok.replace("▁", " ").replace("Ġ", " "), lp)
        for tok, lp in token_logprobs
    ]

    # Aggregate stats
    total_log_prob = sum(lp for _, lp in token_logprobs)
    avg_log_prob = total_log_prob / len(token_logprobs)
    surprisal = -avg_log_prob
    overall_prob = math.exp(total_log_prob)

    return {
        "surprisal": surprisal,
        "avg_log_prob": avg_log_prob,
        "acceptability_prob": overall_prob,
        "log_probs_per_token": token_logprobs,
    }



def analyze_sentences_df(df, model_id=TOGETHER_MODEL, api_key=None):
    print(f"Using Together model: {model_id}")

    client = Together(api_key=api_key)

    


    results = []
    for idx, row in df.iterrows():
        sentence = row['sentence']

        try:
            single_result = get_logprobs(sentence, client, model_id)
        except Exception as e:
            single_result = {
                "surprisal": None,
                "avg_log_prob": None,
                "acceptability_prob": None,
                "log_probs_per_token": None,
                "error": str(e)
            }

        results.append(single_result)

    return pd.DataFrame(results)


In [ ]:

if __name__ == "__main__":
    # Load your dataset
    df = pd.read_csv("exp3_ST.csv")

    # Run all local models
    for model_name in TOGETHER_MODEL:
       

        # Invoke garbage collector
        gc.collect()

        # 3. Clear GPU cache
        # torch.cuda.empty_cache()
        print(f"\n=== Analyzing with model: {model_name} ===")
        results_df = analyze_sentences_df(df, model_name)
        results_df=pd.concat((df, results_df), axis=1)
        results_df.to_csv(f"results/results_{model_name.replace('/', '_')}.csv", index=False)
        print(f"Results saved for {model_name}")
        


=== Analyzing with model: meta-llama/Llama-3.3-70B-Instruct-Turbo ===
Using Together model: meta-llama/Llama-3.3-70B-Instruct-Turbo
Results saved for meta-llama/Llama-3.3-70B-Instruct-Turbo
